# 📘 SHAPE Agent 16 — Section 1: Learning Design
## Authored by: **Cherluk**
### ITAI2376 · Section 14562 · Houston Community College

---

| | |
|---|---|
| **Author** | Cherluk |
| **Role** | Learning Design & Instructional Experience |
| **Design Contribution** | Problem definition, curriculum structure, lesson authoring, Lesson Builder UX |
| **Coding** | Curriculum seeding functions, lesson validation, content authoring cells |
| **Reflection** | See Section 5 of this notebook |

**Full team:** Michael Rivas · Cherluk · Olu · Taki

---

## What this notebook covers

This is **Section 1 of 4** in the SHAPE Agent 16 development series.  
I defined the problem, designed the curriculum schema, and authored the starter lessons  
that the AI tutor draws from when answering learner questions.

**Handoff:** My output (populated curriculum DB) is consumed by Olu in Section 2.

---

**Filename:** `NB_Cherluk_Section1_LearningDesign_Group_ITAI2376_14562.ipynb`

In [ ]:
# ── Setup (run first) ─────────────────────────────────────────────────────────
!pip install -q groq gradio python-dotenv

import os, sys
from pathlib import Path

REPO_URL = 'https://github.com/YOUR_ORG/SHAPE_Agent16.git'  # Michael updates this
REPO_DIR = '/content/SHAPE_Agent16'
if not Path(REPO_DIR).exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --quiet
os.chdir(REPO_DIR)
sys.path.insert(0, 'src')
print('✅ Ready.')

---
## 1. Problem Statement

**Author: Cherluk**

Community college students learning Python and AI face three persistent barriers:

1. **Uneven prior knowledge** — a single lecture pace leaves fast learners bored and slow learners lost.
2. **Delayed feedback** — graded assignments return days later, long after the learning moment has passed.
3. **Static curriculum** — instructors cannot push targeted lessons mid-week without a full deployment pipeline.

**SHAPE Agent 16** solves all three:
- Detects each learner's level and adjusts its explanation style in real time
- Returns instant feedback on code submissions and answers
- Provides a no-code web form so any team member can publish new lessons immediately

**Domain:** Programming education (Python + AI concepts) — directly aligned with ITAI2376 course material.

**Project Option:** Option 4 — Interactive Learning Companion

| Required Feature | How SHAPE Agent 16 delivers it |
|---|---|
| Learner profile building | SQLite `learners` table — level, topics seen, updated each turn |
| Personalised content generation | System prompt dynamically formatted with `{level}` and `{strategy}` |
| Progress assessment and feedback | Pre/post assessments + per-turn ✅/⚠️/💡 structured feedback |
| Learning path adjustment | Automatic level promotion after 3 consecutive perfect-score turns |

In [ ]:
# ── 2. Database Initialisation ────────────────────────────────────────────────
# Cherluk: I initialise the database that will hold all curriculum content.

import shape_db_handler as db
db.init_db()
print('✅ Database initialised.')
print(f'   Location: data/shape.db')

In [ ]:
# ── 3. Curriculum Design & Lesson Authoring ───────────────────────────────────
# Cherluk: I designed the lesson schema and authored the foundational curriculum.
# Each lesson has: topic, level, title, content, keywords, author.
# The keywords field is what Tool 2 (Olu's retrieval tool) searches against.

from shape_safety import validate_lesson_content

MY_LESSONS = [
    {
        'topic': 'Python Basics', 'level': 'beginner',
        'title': 'Variables and Data Types',
        'content': (
            'A variable stores a value in memory. Python infers the type automatically.\n\n'
            'name = "Alice"   # str\n'
            'age  = 21        # int\n'
            'gpa  = 3.8       # float\n'
            'enrolled = True  # bool\n\n'
            'Practice: Create one variable of each type and print them.'
        ),
        'keywords': ['variable','variables','int','float','str','bool','type','data'],
        'author': 'Cherluk',
    },
    {
        'topic': 'Python Basics', 'level': 'beginner',
        'title': 'For Loops and Iteration',
        'content': (
            'A for loop repeats a block of code for each item in a sequence.\n\n'
            'for i in range(5):  # prints 0 1 2 3 4\n'
            '    print(i)\n\n'
            'You can also iterate over a list:\n'
            'fruits = ["apple", "banana"]\n'
            'for fruit in fruits:\n'
            '    print(fruit)\n\n'
            'Practice: Write a loop that prints the squares of numbers 1 through 10.'
        ),
        'keywords': ['loop','loops','for','iteration','range','iterate'],
        'author': 'Cherluk',
    },
    {
        'topic': 'Python Basics', 'level': 'beginner',
        'title': 'If / Else Decisions',
        'content': (
            'Use if/else to make your program choose between paths.\n\n'
            'score = 85\n'
            'if score >= 90:\n'
            '    grade = "A"\n'
            'elif score >= 80:\n'
            '    grade = "B"\n'
            'else:\n'
            '    grade = "C"\n\n'
            'Practice: Write an if/else that prints Pass or Fail based on a score variable.'
        ),
        'keywords': ['if','else','elif','condition','conditional','decision','branch'],
        'author': 'Cherluk',
    },
    {
        'topic': 'Python Basics', 'level': 'beginner',
        'title': 'Functions',
        'content': (
            'A function is a reusable block of code defined with def.\n\n'
            'def greet(name):\n'
            '    return f"Hello, {name}!"\n\n'
            'print(greet("World"))  # Hello, World!\n\n'
            'Default parameters:\n'
            'def power(base, exp=2):\n'
            '    return base ** exp\n\n'
            'Practice: Write a function that returns the average of a list of numbers.'
        ),
        'keywords': ['function','functions','def','return','parameter','call','argument'],
        'author': 'Cherluk',
    },
    {
        'topic': 'AI Concepts', 'level': 'intermediate',
        'title': 'Supervised vs Unsupervised Learning',
        'content': (
            'Supervised Learning: data has labels (input→output pairs).\n'
            '  Examples: classification, regression\n'
            '  Algorithms: decision trees, SVM, neural networks\n\n'
            'Unsupervised Learning: data has NO labels.\n'
            '  Examples: clustering, dimensionality reduction\n'
            '  Algorithms: K-Means, PCA, autoencoders\n\n'
            'Key difference: presence of labelled training data.'
        ),
        'keywords': ['supervised','unsupervised','learning','classification','clustering','ml','machine'],
        'author': 'Cherluk',
    },
]

saved = 0
for lesson in MY_LESSONS:
    v = validate_lesson_content(lesson['title'], lesson['content'], lesson['keywords'])
    if not v.safe:
        print(f'❌ {lesson["title"]}: {v.reason}')
        continue
    existing = db.search_lessons(lesson['keywords'], level=lesson['level'])
    if lesson['title'] not in [e['title'] for e in existing]:
        lid = db.add_lesson(**lesson)
        print(f'✅ Saved [{lid}]: {lesson["title"]} (by {lesson["author"]})')
        saved += 1
    else:
        print(f'⏭  Already exists: {lesson["title"]}')

print(f'\n{saved} new lessons saved to curriculum database.')

In [ ]:
# ── 4. Add Your Own Lesson (Live Authoring Demo) ──────────────────────────────
# Cherluk: This demonstrates the live curriculum authoring feature I designed.
# Any team member can add a lesson here — no Python knowledge required.
# Just fill in the fields below and run the cell.

NEW_LESSON = {
    'topic':    'Python Basics',
    'level':    'beginner',
    'title':    'Lists — Creating and Accessing',
    'content':  (
        'A list stores multiple values in a single variable.\n\n'
        'fruits = ["apple", "banana", "cherry"]\n'
        'print(fruits[0])   # apple  (index starts at 0)\n'
        'print(fruits[-1])  # cherry (last element)\n\n'
        'Common list methods:\n'
        '  fruits.append("mango")   # add to end\n'
        '  fruits.remove("banana") # remove by value\n'
        '  len(fruits)              # number of items\n\n'
        'Practice: Create a list of 5 numbers and print their sum using sum().'
    ),
    'keywords': ['list','lists','append','index','remove','len','items'],
    'author':   'Cherluk',
}

v = validate_lesson_content(NEW_LESSON['title'], NEW_LESSON['content'], NEW_LESSON['keywords'])
if v.safe:
    lid = db.add_lesson(**NEW_LESSON)
    print(f'✅ Lesson "{NEW_LESSON["title"]}" saved (ID: {lid})')
    print('The agent will use this lesson immediately when learners ask about lists.')
else:
    print(f'❌ Validation error: {v.reason}')

In [ ]:
# ── 5. Verify Curriculum ──────────────────────────────────────────────────────
all_lessons = db.search_lessons(
    ['python','loop','function','variable','if','list','supervised'], level=None)
print(f'Curriculum database — {len(all_lessons)} lessons total:')
print()
for l in all_lessons:
    print(f'  [{l["level"]:12s}] {l["title"]:40s} by {l["author"]}')

---
## 6. Reflection — Cherluk

**Design decisions I made:**

- **Keyword-based schema** — I chose to tag each lesson with a `keywords` list rather than a category tree. This lets the retrieval tool (Olu's Tool 2) find lessons from natural language queries without needing a vector database.
- **Level stratification** — Separating `beginner / intermediate / advanced` means the agent never serves advanced content to a learner who hasn't demonstrated readiness, satisfying the Option 4 requirement for learning path adjustment.
- **Author attribution** — Every lesson stores the author's name. This makes it easy to see who contributed what content and supports team accountability.

**Challenges I encountered:**

- Deciding how much detail to put in each lesson — too short and the agent has nothing to work with; too long and it overflows the LLM context window. I settled on 150–300 words per lesson.
- Ensuring keywords were specific enough to avoid false matches (e.g., 'range' only in loop lessons, not general ones).

**What I learned:**

- Curriculum design for AI systems is different from writing a textbook — you're writing for a retrieval index, not a human reader.
- The `validate_lesson_content()` safety check I discovered in Olu's safety module protects the DB from empty or malformed entries.

---

## ✋ Handoff to Olu (Section 2)

**Delivered:**
- Problem statement and Option 4 alignment table
- 6 lessons in the curriculum DB (variables, loops, if/else, functions, lists, supervised learning)
- Live lesson authoring demonstration

**Olu needs to:** Build the tools that let the agent search this curriculum (Tool 2) and safely execute student code (Tool 1), plus the safety layer that protects the system.

---
*SHAPE Agent 16 — ITAI2376_14562 | Michael Rivas · **Cherluk** · Olu · Taki*